### RAG 

In [68]:
from langchain_core.documents import Document

In [69]:
doc=Document(
    page_content="Hello World",
    metadata={
        "source":"https://www.google.com"
    }
)

In [70]:
doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World')

In [71]:
type(doc)

langchain_core.documents.base.Document

In [72]:
# Text data

# from langchain_community.document_loaders.text import TextLoader

# loader=TextLoader("data/python.txt",encoding="utfa8")

# document=loader.load()

# document

In [73]:
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader=PyPDFLoader("data/research1.pdf")

# document=pdf_loader.load()

# document


### Ingestion Pipeline

In [74]:
from langchain_community.document_loaders.pdf import PyPDFLoader
import os

### Documents

In [75]:
def load_all_pdfs():
    folder_path="data/pdfs"
    num_docs=0
    all_docs=[]

    for filename in os.listdir(folder_path):

        if filename.lower().endswith(".pdf"):

            pdf_path=os.path.join(folder_path,filename)

            loader=PyPDFLoader(pdf_path)

            doc=loader.load()

            all_docs.extend(doc)
            num_docs+=1

    print(f"Total Docs:{num_docs}")
    print(f"Total pages:{len(all_docs)}")
    return all_docs

In [76]:
all_pdf_documents=load_all_pdfs()

Total Docs:2
Total pages:32


### Chunking

In [77]:
# Chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [78]:
def split_docs(documents,chunk_size=500,chunk_overlap=50):

    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_docs=text_splitter.split_documents(documents)

    return chunked_docs

In [79]:
chunks=split_docs(all_pdf_documents)

In [80]:
len(chunks)

321

In [81]:
# chunks[0]

### Embedding

In [82]:
from sentence_transformers import SentenceTransformer

In [91]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("Loading Model:",self.model_name)
        self.model=SentenceTransformer(model_name)
        print("Embedding Dimension",self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self,text):
        embeddings=self.model.encode(text,show_progress_bar=True)
        print("Embeddings Model:",embeddings.shape)
        return embeddings

In [92]:
embedding_manager=EmbeddingManager()

Loading Model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Dimension 384


C:\Users\HP\AppData\Local\Temp\ipykernel_13452\704684039.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding Dimension",self.model.get_sentence_embedding_dimension())


### Vector Store

In [93]:
import chromadb
import uuid

In [112]:
class VectorStoreManager:
    def __init__(self,collection_name="pdf_documents",persistent_directory="data/vectorstore"):
        self.collection_name=collection_name
        self.persistent_directory=persistent_directory
        self.collection=None
        self.client=None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persistent_directory,exist_ok=True)

        # make client
        self.client=chromadb.PersistentClient(path=self.persistent_directory)

        # make collection
        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description":"Vector store collection for pdf embeddings in RAG"
            }
        )

        print("Initialized the vector store with collection",self.collection_name)
        print("Docs in collection",self.collection.count())

    def add_documents(self,documents,embeddings):
        if len(documents)!=len(embeddings):
            raise ValueError("No of documents does not match number of embeddings")

        #store => ids,documents,embeddings,metadata

        ids=[]
        all_metadata=[]
        document_content=[]
        embedding_list=[]

        for i,(document,embedding) in enumerate(zip(documents,embeddings)):

            doc_id=f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata["index"]=i
            metadata["Lenght of context"]=len(doc.page_content)
            all_metadata.append(metadata)

            document_content.append(doc.page_content)

            embedding_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=document_content,
            embeddings=embedding_list

        )

        print(f"Total Documents added in vector store:{len(document_content)}")
        print("Docs in collection",self.collection.count())

In [113]:
vectorstore=VectorStoreManager()

Initialized the vector store with collection pdf_documents
Docs in collection 0


In [114]:
vectorstore

### Data => Documents => chunks => embeddings => Store in vector database

In [115]:
texts=[doc.page_content for doc in chunks]

In [116]:
embeddings=embedding_manager.generate_embeddings(texts)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embeddings Model: (321, 384)


In [117]:

vectorstore.add_documents(chunks,embeddings)

Total Documents added in vector store:321
Docs in collection 321
